# Lead Intelligence System - Business Analyst Dashboard

**Phase 1-3 Implementation Review**  
Interactive exploration of data, rubric, scoring, and results.

---

## Quick Start
1. Run each cell in order (Shift+Enter)
2. Modify parameters in **Configuration** cells to explore
3. All visualizations update automatically

**Prerequisites:** Run from `lead-qualifier` directory with venv activated

In [ ]:
# ============================================================
# SETUP: Imports and Path Configuration
# ============================================================
import sys
import os
from pathlib import Path

# Add lead-qualifier to path
PROJECT_ROOT = Path.cwd()
LEAD_QUALIFIER = PROJECT_ROOT / "lead-qualifier"
sys.path.insert(0, str(LEAD_QUALIFIER))

import pandas as pd
import numpy as np
import json
import yaml
from datetime import date, datetime
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 50)
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print(f"Project root: {PROJECT_ROOT}")
print(f"Lead qualifier: {LEAD_QUALIFIER}")
print(f"Exists: {LEAD_QUALIFIER.exists()}")

In [ ]:
# ============================================================
# 1. DATA EXPLORATION
# ============================================================

# Load all three datasets
training_df = pd.read_csv(LEAD_QUALIFIER / ".." / "leads_training.csv")
testing_df = pd.read_csv(LEAD_QUALIFIER / ".." / "leads_testing.csv")
full_df = pd.read_csv(LEAD_QUALIFIER / ".." / "leads_full_scale_100.csv")

print("=== DATASET OVERVIEW ===")
print(f"Training:  {len(training_df)} leads")
print(f"Testing:   {len(testing_df)} leads")
print(f"Full:      {len(full_df)} leads")

print("\n=== TRAINING DATA SAMPLE ===")
display(training_df.head(10))

In [ ]:
# Dataset statistics
print("=== TRAINING DATA STATS ===")
print(f"Shape: {training_df.shape}")
print(f"\nColumns: {list(training_df.columns)}")
print(f"\nDtypes:\n{training_df.dtypes}")
print(f"\nMissing values:\n{training_df.isnull().sum()}")
print(f"\nCompany size stats:\n{training_df['company_size'].describe()}")
print(f"\nIndustry distribution (top 10):\n{training_df['industry'].value_counts().head(10)}")
print(f"\nSource distribution:\n{training_df['source'].value_counts()}")
print(f"\nDate range: {training_df['last_interaction_date'].min()} to {training_df['last_interaction_date'].max()}")

In [ ]:
# Testing data - edge cases
print("=== TESTING DATA - EDGE CASES ===")
print(f"Shape: {testing_df.shape}")
print(f"\nMissing values:\n{testing_df.isnull().sum()}")
print(f"\nCompany size unique (including NaN):\n{testing_df['company_size'].value_counts(dropna=False)}")
print(f"\nIndustry unique:\n{testing_df['industry'].value_counts(dropna=False)}")
print(f"\nSource unique:\n{testing_df['source'].value_counts(dropna=False)}")

# Show rows with missing data
edge_cases = testing_df[testing_df.isnull().any(axis=1)]
print(f"\nRows with missing data ({len(edge_cases)}):")
display(edge_cases)

In [ ]:
# Full scale data overview
print("=== FULL SCALE DATA (100 leads) ===")
print(f"Shape: {full_df.shape}")
print(f"\nMissing values:\n{full_df.isnull().sum()}")
print(f"\nCompany size stats:\n{full_df['company_size'].describe()}")
print(f"\nIndustry count: {full_df['industry'].nunique()} unique")
print(f"\nSource distribution:\n{full_df['source'].value_counts()}")
print(f"\nDate range: {full_df['last_interaction_date'].min()} to {full_df['last_interaction_date'].max()}")

In [ ]:
# ============================================================
# 2. RUBRIC CONFIGURATION VIEWER
# ============================================================

with open(LEAD_QUALIFIER / "config.yaml") as f:
    config = yaml.safe_load(f)

rubric = config["rubric"]
thresholds = config["thresholds"]
llm_config = config["llm"]

print("=== RUBRIC FACTORS & WEIGHTS ===")
rubric_summary = []
for name, factor in rubric.items():
    rubric_summary.append({
        "Factor": name.replace("_", " ").title(),
        "Weight": f"{factor['weight']*100:.0f}%",
        "Tiers": len(factor["tiers"]),
        "Default": factor["tiers"][-1].get("default", "N/A")
    })
display(pd.DataFrame(rubric_summary))

In [ ]:
# Detailed tier breakdown for each factor
for name, factor in rubric.items():
    print(f"\n=== {name.replace('_', ' ').title()} (Weight: {factor['weight']*100:.0f}%) ===")
    tiers_data = []
    for i, tier in enumerate(factor["tiers"]):
        if "range" in tier:
            desc = f"{tier['range'][0]} - {tier['range'][1]} employees"
        elif "days" in tier:
            desc = f"{tier['days'][0]} - {tier['days'][1]} days ago"
        elif "industries" in tier:
            desc = ", ".join(tier["industries"])
        elif "sources" in tier:
            desc = ", ".join(tier["sources"])
        elif "condition" in tier:
            desc = f"Condition: {tier['condition'][:80]}..."
        else:
            desc = "Default"
        tiers_data.append({
            "Tier": i+1,
            "Criteria": desc,
            "Score": tier.get("score", tier.get("default", "N/A"))
        })
    display(pd.DataFrame(tiers_data))

In [ ]:
print("=== DECISION THRESHOLDS ===")
threshold_df = pd.DataFrame([
    {"Decision": "QUALIFIED", "Min Score": thresholds["qualified"], "Max Score": 100},
    {"Decision": "REVIEW", "Min Score": thresholds["review"], "Max Score": thresholds["qualified"]-1},
    {"Decision": "REJECTED", "Min Score": 0, "Max Score": thresholds["review"]-1}
])
display(threshold_df)

print("=== LLM CONFIGURATION ===")
display(pd.DataFrame([llm_config]).T.rename(columns={0: "Value"}))

In [ ]:
# ============================================================
# 3. SCORING ENGINE DEMO - Interactive Single Lead Scoring
# ============================================================

from utils.data_loader import load_leads
from utils.rubric_engine import load_rubric, score_lead
from utils.models import Lead

rubric = load_rubric(str(LEAD_QUALIFIER / "config.yaml"))

def score_single_lead(lead_data, reference_date=date(2024, 1, 20)):
    """Score a single lead and return detailed breakdown."""
    lead = Lead.from_dict(lead_data)
    score = score_lead(lead, rubric, reference_date)
    
    result = {
        "lead": {
            "name": lead.name,
            "company": lead.company,
            "company_size": lead.company_size,
            "industry": lead.industry,
            "source": lead.source,
            "last_interaction_date": str(lead.last_interaction_date) if lead.last_interaction_date else None
        },
        "total_score": score.total_score,
        "decision": score.decision,
        "priority": score.priority,
        "factor_breakdown": {}
    }
    for fs in score.factor_scores:
        result["factor_breakdown"][fs.name] = {
            "raw_score": fs.raw_score,
            "weight": fs.weight,
            "weighted_score": round(fs.weighted_score, 2),
            "contribution_pct": round(fs.weighted_score / sum(f.weighted_score for f in score.factor_scores) * 100, 1)
        }
    return result

# Demo: Score first 3 training leads
leads = load_leads(str(LEAD_QUALIFIER / ".." / "leads_training.csv"))

for lead in leads[:3]:
    result = score_single_lead({
        "name": lead.name,
        "company": lead.company,
        "company_size": lead.company_size,
        "industry": lead.industry,
        "source": lead.source,
        "last_interaction_date": lead.last_interaction_date
    })
    print(f"\n{'='*60}")
    print(f"{result['lead']['name']} - {result['lead']['company']}")
    print(f"Total Score: {result['total_score']} | Decision: {result['decision']} | Priority: {result['priority']}")
    print(f"Factor Breakdown:")
    for fname, fdata in result["factor_breakdown"].items():
        print(f"  {fname}: raw={fdata['raw_score']}, weighted={fdata['weighted_score']}, contrib={fdata['contribution_pct']}%")

In [ ]:
# ============================================================
# 4. BATCH SCORING RESULTS - Training Data
# ============================================================

from utils.rubric_engine import score_leads_batch

leads = load_leads(str(LEAD_QUALIFIER / ".." / "leads_training.csv"))
scores = score_leads_batch(leads, rubric, reference_date=date(2024, 1, 20))

# Convert to DataFrame for analysis
results = []
for s in scores:
    row = {
        "name": s.lead.name,
        "company": s.lead.company,
        "company_size": s.lead.company_size,
        "industry": s.lead.industry,
        "source": s.lead.source,
        "last_interaction_date": s.lead.last_interaction_date,
        "total_score": s.total_score,
        "decision": s.decision,
        "priority": s.priority
    }
    for fs in s.factor_scores:
        row[f"factor_{fs.name}"] = fs.raw_score
    results.append(row)

results_df = pd.DataFrame(results)

print("=== TRAINING DATA - SCORING SUMMARY ===")
print(f"Total leads: {len(results_df)}")
print(f"\nDecision distribution:")
print(results_df['decision'].value_counts())
print(f"\nPriority distribution:")
print(results_df['priority'].value_counts())
print(f"\nScore statistics:")
print(results_df['total_score'].describe())

display(results_df[['name', 'company', 'total_score', 'decision', 'priority']].head(15))

In [ ]:
# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Score distribution
axes[0, 0].hist(results_df['total_score'], bins=20, edgecolor='black', alpha=0.7)
axes[0, 0].axvline(x=75, color='green', linestyle='--', label='Qualified (75)')
axes[0, 0].axvline(x=50, color='orange', linestyle='--', label='Review (50)')
axes[0, 0].set_xlabel('Total Score')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('Score Distribution (Training)')
axes[0, 0].legend()

# Decision pie
decision_counts = results_df['decision'].value_counts()
colors = ['#2ecc71', '#f39c12', '#e74c3c']
axes[0, 1].pie(decision_counts.values, labels=decision_counts.index, autopct='%1.1f%%', colors=colors)
axes[0, 1].set_title('Decision Distribution')

# Score by industry (top 10)
industry_scores = results_df.groupby('industry')['total_score'].mean().sort_values(ascending=False).head(10)
industry_scores.plot(kind='barh', ax=axes[1, 0], color='steelblue')
axes[1, 0].set_xlabel('Avg Score')
axes[1, 0].set_title('Avg Score by Industry (Top 10)')
axes[1, 0].invert_yaxis()

# Score by source
source_scores = results_df.groupby('source')['total_score'].mean().sort_values(ascending=False)
source_scores.plot(kind='bar', ax=axes[1, 1], color='coral')
axes[1, 1].set_ylabel('Avg Score')
axes[1, 1].set_title('Avg Score by Source')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Factor contribution analysis
factor_cols = [c for c in results_df.columns if c.startswith('factor_')]
factor_means = results_df[factor_cols].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
factor_means.plot(kind='bar', color='mediumpurple')
plt.title('Average Raw Factor Scores (1-10 scale)')
plt.ylabel('Average Raw Score')
plt.xlabel('Factor')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Average factor scores:")
for col, val in factor_means.items():
    print(f"  {col}: {val:.2f}")

In [ ]:
# ============================================================
# 5. OUTPUT REPORT VIEWER
# ============================================================

def load_report(report_path):
    with open(report_path) as f:
        return json.load(f)

# Load all three reports
training_report = load_report(LEAD_QUALIFIER / "output_training.json")
testing_report = load_report(LEAD_QUALIFIER / "output_testing.json")
full_report = load_report(LEAD_QUALIFIER / "output_report.json")

print("=== REPORT METADATA ===")
for name, report in [("Training", training_report), ("Testing", testing_report), ("Full", full_report)]:
    meta = report["metadata"]
    print(f"\n{name} Report:")
    for k, v in meta.items():
        print(f"  {k}: {v}")

In [ ]:
print("=== SUMMARY COMPARISON ===")
summary_data = []
for name, report in [("Training", training_report), ("Testing", testing_report), ("Full", full_report)]:
    s = report["summary"]
    summary_data.append({
        "Dataset": name,
        "Total": s["total_processed"],
        "Qualified": s["qualified_count"],
        "Review": s["review_count"],
        "Rejected": s["rejected_count"],
        "Qualified %": s["qualified_percentage"],
        "Flagged": len(report["flagged_for_review"]),
        "Errors": len(report["errors"])
    })
display(pd.DataFrame(summary_data))

In [ ]:
# Full report - priority queue (top 20)
print("=== FULL SCALE - PRIORITY QUEUE (TOP 20) ===")
pq_df = pd.DataFrame(full_report["priority_queue"])
display(pq_df.head(20))

In [ ]:
# Full report - sample outreach messages
print("=== SAMPLE OUTREACH MESSAGES ===")
for msg in full_report["sample_outreach_messages"]:
    print(f"\n--- {msg['lead']} ---")
    print(msg['message'][:300])
    print("...")

In [ ]:
# Full report - flagged for review
print("=== FLAGGED FOR REVIEW ===")
if full_report["flagged_for_review"]:
    flagged_df = pd.DataFrame(full_report["flagged_for_review"])
    display(flagged_df)
else:
    print("No flagged leads in full scale report (all 100 records complete)")

# Testing report - flagged for review
print("\n=== TESTING REPORT - FLAGGED FOR REVIEW ===")
if testing_report["flagged_for_review"]:
    flagged_df = pd.DataFrame(testing_report["flagged_for_review"])
    display(flagged_df)

In [ ]:
# ============================================================
# 6. INTERACTIVE FILTERING & EXPLORATION
# ============================================================

# Full scale leads detail
full_leads_df = pd.DataFrame(full_report["leads"])

print("=== INTERACTIVE EXPLORATION ===")
print(f"Total leads: {len(full_leads_df)}")
print(f"Columns: {list(full_leads_df.columns)}")

# Filter examples
print("\n--- Qualified Leads ---")\nqualified = full_leads_df[full_leads_df['decision'] == 'QUALIFIED']
print(f"Count: {len(qualified)}")
display(qualified[['name', 'company', 'company_size', 'industry', 'source', 'score', 'priority']].head(15))

In [ ]:
# Filter by industry
print("--- Leads by Industry (Qualified only) ---")
if len(qualified) > 0:
    industry_qual = qualified['industry'].value_counts()
    display(industry_qual.to_frame('Qualified Count'))

In [ ]:
# Filter by company size range
print("--- Score Distribution by Company Size Bands ---")
full_leads_df['size_band'] = pd.cut(full_leads_df['company_size'], 
    bins=[0, 10, 50, 500, 5000, 10000, float('inf')],
    labels=['1-10', '11-50', '51-500', '501-5000', '5001-10000', '10000+'])

size_band_stats = full_leads_df.groupby('size_band', observed=False).agg(
    count=('score', 'count'),
    avg_score=('score', 'mean'),
    qualified=('decision', lambda x: (x == 'QUALIFIED').sum())
).round(1)
display(size_band_stats)

In [ ]:
# Factor breakdown for qualified vs rejected
print("=== FACTOR SCORES: QUALIFIED vs REJECTED ===")

factor_cols = [c for c in full_leads_df.columns if c.startswith('factor_')]
if factor_cols:
    qualified_factors = full_leads_df[full_leads_df['decision'] == 'QUALIFIED'][factor_cols].mean()
    rejected_factors = full_leads_df[full_leads_df['decision'] == 'REJECTED'][factor_cols].mean()
    review_factors = full_leads_df[full_leads_df['decision'] == 'REVIEW'][factor_cols].mean()
    
    comparison = pd.DataFrame({
        'Qualified': qualified_factors,
        'Review': review_factors,
        'Rejected': rejected_factors
    }).round(2)
    display(comparison)
    
    # Plot
    comparison.plot(kind='bar', figsize=(10, 5))
    plt.title('Average Factor Scores by Decision')
    plt.ylabel('Raw Score (1-10)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# 7. EDGE CASE ANALYSIS
# ============================================================

print("=== TESTING DATA - EDGE CASE SCORING ===")

testing_leads = load_leads(str(LEAD_QUALIFIER / ".." / "leads_testing.csv"))
testing_scores = score_leads_batch(testing_leads, rubric, reference_date=date(2024, 1, 20))

for s in testing_scores:
    issues = []
    if s.lead.company_size is None: issues.append("No size")
    if not s.lead.industry: issues.append("No industry")
    if not s.lead.source: issues.append("No source")
    if s.lead.last_interaction_date is None: issues.append("No date")
    
    if issues:
        print(f"\n{s.lead.name} ({s.lead.company}):")
        print(f"  Issues: {', '.join(issues)}")
        print(f"  Score: {s.total_score} | Decision: {s.decision} | Priority: {s.priority}")
        for fs in s.factor_scores:
            print(f"  {fs.name}: raw={fs.raw_score}, weighted={fs.weighted_score:.2f}")

In [ ]:
# ============================================================
# 8. EXPORT & SHARING
# ============================================================

# Export full results to CSV for Excel/BI tools
full_leads_df.to_csv(LEAD_QUALIFIER / "full_scale_results.csv", index=False)
results_df.to_csv(LEAD_QUALIFIER / "training_results.csv", index=False)

print("Exported:")
print(f"  - {LEAD_QUALIFIER / 'full_scale_results.csv'}")
print(f"  - {LEAD_QUALIFIER / 'training_results.csv'}")

# Export summary for presentation
summary_export = {
    "project": "Lead Intelligence System",
    "phase": "Phase 3 Complete",
    "rubric": {name: {"weight": f["weight"], "tiers": len(f["tiers"])} for name, f in rubric.items()},
    "thresholds": thresholds,
    "training_results": training_report["summary"],
    "testing_results": testing_report["summary"],
    "full_scale_results": full_report["summary"],
    "top_10_leads": full_report["priority_queue"][:10]
}

with open(LEAD_QUALIFIER / "phase3_summary.json", "w") as f:
    json.dump(summary_export, f, indent=2, default=str)

print(f"  - {LEAD_QUALIFIER / 'phase3_summary.json'}")

print("\n✅ Dashboard complete! Use exported CSVs in Excel, Tableau, or Power BI.")